### 1. What is tokenization?

In [24]:
text = 'hello world'
vocab = text.split()
id = [vocab.index(word) for word in vocab]
print(vocab)
print(id)

print(vocab.index('world'))

['hello', 'world']
[0, 1]
1


#### Build a character level tokenization

In [55]:
text = 'hello world, we are living in a different world'
vocab = sorted(set(text))
stoi = {char:i for i,char in enumerate(vocab)}
itos = {i:char for char, i in stoi.items()}

def encode(text):
    tokens = [stoi[char] for char in text]
    return tokens

def decode(tokens):
    chars = [itos[token] for token in tokens]
    return ''.join(chars)

text = 'hello world'
# print(stoi, itos)
# print(text)
# print(encode(text))
# print(decode(encode(text)))

test_texts = [
    "hello",
    "hello,",
    "hello world",
    "different world",
    "new set of word"
]

for text in test_texts:
    # print(f'{text} -> {encode(text)} -> {decode(encode(text))}')
    if decode(encode(text)) == text:
        print(f'{text} is preserved!')


hello is preserved!
hello, is preserved!
hello world is preserved!
different world is preserved!


KeyError: 's'

### Observation

The character tokenizer can only encode characters that were present
when the vocabulary was created.

This creates an unknown-character problem.

In [92]:
import re
text = 'hello world,   we are living in a different world'

tokens = re.split(r"(,)| +", text)
vocab = sorted(set([token for token in tokens if token and token.split() != '']))
stoi = {word:i for i, word in enumerate(vocab)}
itos = {i:word for word, i in stoi.items()}

def encode(text):
    tokens = re.split(r"(,)| +", text)
    return [stoi[token] for token in tokens if token and token.split() != '']

def decode(tokens):
    words = [itos[token] for token in tokens]
    return ' '.join(words)

test_texts = [
    "hello",
    "hello,",
    "hello world",
    "different world",
    "hello world I am king"
]

for text in test_texts:
    # print(f'{text} -> {encode(text)} -> {decode(encode(text))}')
    if decode(encode(text)) == text:
        print(f'{text} is preserved!')
    else:
        print(f'{text} is not preserved!')


hello is preserved!
hello, is not preserved!
hello world is preserved!
different world is preserved!


KeyError: 'I'

In [24]:
# let's add <unk> in vocab for unknown word
import re
text = 'hello world, we are living in a different world'

tokens = re.split(r"(,)| +", text)
vocab = sorted(set([token for token in tokens if token and token.split() != '']))
stoi = {word:i for i, word in enumerate(vocab)}
stoi['<unk>'] = len(stoi)
itos = {i:word for word, i in stoi.items()}
itos[len(stoi)-1] = '<unk>'

def encode(text):
    tokens = re.split(r"(,)| +", text)
    return [stoi[token] if token in stoi else stoi['<unk>'] for token in tokens if token and token.split() != '']

def decode(tokens):
    words = [itos[token] for token in tokens]
    return ' '.join(words)

test_texts = [
    "hello",
    "hello,",
    "hello world",
    "different world",
    'hello world, I am king world'
]

for text in test_texts:
    print(f'{text} -> {encode(text)} -> {decode(encode(text))}')
    if decode(encode(text)) == text:
        print(f'{text} is preserved!')
    else:
        print(f'{text} is not preserved!')


hello -> [4] -> hello
hello is preserved!
hello, -> [4, 0] -> hello ,
hello, is not preserved!
hello world -> [4, 8] -> hello world
hello world is preserved!
different world -> [3, 8] -> different world
different world is preserved!
hello world, I am king world -> [4, 8, 0, 9, 9, 9, 8] -> hello world , <unk> <unk> <unk> world
hello world, I am king world is not preserved!


### Observation: Unknown words

Using `<UNK>` allows the tokenizer to handle words that were not present
in the vocabulary.

However, all unknown words are mapped to the same token. This means the
original word cannot be recovered during decoding.

For example, different unseen words such as `king`, `queen`, and
`astronaut` would all become `<UNK>`.

In [ ]:
# let's play with some more word variations in vocab and see the pattern 
import re
text = 'hello play, we are playing and player played like a king in the playground and was unhappy but happiness'

tokens = re.split(r"(,)| +", text)
vocab = sorted(set([token for token in tokens if token and token.split() != '']))
stoi = {word:i for i, word in enumerate(vocab)}
stoi['<unk>'] = len(stoi)
itos = {i:word for word, i in stoi.items()}
itos[len(stoi)-1] = '<unk>'

def encode(text):
    tokens = re.split(r"(,)| +", text)
    return [stoi[token] if token in stoi else stoi['<unk>'] for token in tokens if token and token.split() != '']

def decode(tokens):
    words = [itos[token] for token in tokens]
    return ' '.join(words)

test_texts = [
    "play",
    "playing",
    "played",
    "player",
    "playground",
    "unhappy",
    "happiness",
]
print(stoi)
for text in test_texts:
    # print(f'{text} -> {encode(text)} -> {decode(encode(text))}')
    if decode(encode(text)) == text:
        print(f'{text} is preserved!')
    else:
        print(f'{text} is not preserved!')


{',': 0, 'a': 1, 'and': 2, 'are': 3, 'but': 4, 'happiness': 5, 'hello': 6, 'in': 7, 'king': 8, 'like': 9, 'play': 10, 'played': 11, 'player': 12, 'playground': 13, 'playing': 14, 'the': 15, 'unhappy': 16, 'was': 17, 'we': 18, '<unk>': 19}
play is preserved!
playing is preserved!
played is preserved!
player is preserved!
playground is preserved!
unhappy is preserved!
happiness is preserved!


## From Word Tokenization to Subword Tokenization

Our word-level tokenizer gives us a much smaller sequence compared with
character-level tokenization, but it introduces another problem.

Every complete word is treated as an independent token.

For example:

- `play`
- `playing`
- `played`
- `player`
- `playground`
- `plays`

would each require their own vocabulary entry.

This can make the vocabulary very large because natural language contains
many word variations, inflections, names, rare words, and newly encountered
words.

We also saw the unknown-word problem. If a word was not present when the
vocabulary was created, the tokenizer cannot represent it directly. Using
an `<unk>` token solves the lookup error, but loses information because
different unknown words all become the same token.

This led to an important question:

> Can we represent words using smaller, reusable pieces instead of treating
> every complete word as an indivisible unit?

For example, conceptually:

    playing   → play + ing
    played    → play + ed
    player    → play + er
    plays     → play + s
    playground → play + ground

A vocabulary containing reusable pieces could therefore represent many
different words without requiring every complete word to exist in the
vocabulary.

This is the motivation for **subword tokenization**.

One important distinction is that we should not manually assume that
linguistic units such as `play`, `ing`, or `ground` are always the correct
tokens. A subword algorithm such as BPE learns which pieces are useful from
the data.

The next step is therefore to first experiment with a small manually chosen
subword vocabulary, and then understand how BPE can automatically discover
such reusable pieces.

### Let's try manual sub word level implementation and see

In [41]:
subword_vocab = [
    "play",
    "ing",
    "ed",
    "er",
    "s",
    "ground",
    "un",
    "happy",
    "ness",
]

stoi = {piece:i for i, piece in enumerate(subword_vocab)}
stoi['<unk>'] = len(stoi)
itos = {i:piece for piece, i in stoi.items()}
itos[len(stoi)-1] = '<unk>'

def encode(pieces):
    tokens = [stoi[piece] if piece in stoi else stoi['<unk>'] for piece in pieces]
    return tokens

def decode(tokens):
    words = [itos[t] for t in tokens]
    return ''.join(words)

pieces = [['play', 'er', 's'],
          ['play', 'ing'],
          ['play', 'ed'],
          ['play', 'ground'],
          ['un', 'happy', 'ness'],
          ['un', 'happi', 'ness'],
          ['play', 'in', 'g']
          ]
for piece in pieces:
    print(f'{''.join(piece)} -> {piece} -> {encode(piece)} -> {decode(encode(piece))}')
# encoded_token = encode(pieces)
# decoded_text = decode(encoded_token)
print('stoi:', stoi)
# print(f'{word} -> {pieces} -> {encoded_token} -> {decoded_text}')

players -> ['play', 'er', 's'] -> [0, 3, 4] -> players
playing -> ['play', 'ing'] -> [0, 1] -> playing
played -> ['play', 'ed'] -> [0, 2] -> played
playground -> ['play', 'ground'] -> [0, 5] -> playground
unhappyness -> ['un', 'happy', 'ness'] -> [6, 7, 8] -> unhappyness
unhappiness -> ['un', 'happi', 'ness'] -> [6, 9, 8] -> un<unk>ness
playing -> ['play', 'in', 'g'] -> [0, 9, 9] -> play<unk><unk>
stoi: {'play': 0, 'ing': 1, 'ed': 2, 'er': 3, 's': 4, 'ground': 5, 'un': 6, 'happy': 7, 'ness': 8, '<unk>': 9}


## Subword Tokenization: Reusable Pieces

Instead of treating every complete word as a separate token, I experimented
with representing words using smaller reusable pieces.

For example:

- `players` → `play + er + s`
- `playing` → `play + ing`
- `played` → `play + ed`
- `playground` → `play + ground`

This allows the same vocabulary pieces to represent multiple words.

However, our vocabulary was manually created and very small, so words
containing unseen pieces still produced `<unk>`.

This leads to the next question:

> How can we automatically discover useful reusable subword pieces from a
> corpus instead of manually defining them?

This is the motivation for **Byte Pair Encoding (BPE)**.

In [192]:
# let's understand the algorithm manually
corpus = [
    "low",
    "lower",
    "lowest",
    "low",
    "lower",
]

vocab = sorted(set(''.join(corpus)))
words = []
for word in corpus:
    w = []
    for char in word:
        w.append(char)
    words.append(w)

def get_pair_counts(words):
    pairs = {} 
    for word in words:
        for i,j in zip(word, word[1:]):
            pairs[(i,j)] = pairs.get((i,j), 0) + 1
    # print('pairs:', pairs)
    return pairs

def get_best_pair(pair_counts: dict):
    return max(pair_counts.items(), key=lambda item: item[1])[0]

def merge_pair(words, pair):
    new_words = []
    for word in words:
        merged = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == pair:
                merged.append(word[i]+word[i+1])
                i += 2
            else:
                merged.append(word[i])
                i += 1
        new_words.append(merged)
    return new_words
            
num_merges = 4

def train(words, num_merges: int):
    merges = []
    for _ in range(num_merges):
        pair_counts = get_pair_counts(words)
        best_pair = get_best_pair(pair_counts)
        merges.append(best_pair)
        words = merge_pair(words, best_pair)
        vocab.append(''.join(best_pair))
    return merges, vocab

merges, vocab = train(words, num_merges)
print('merges:', merges)
print('final vocab:', vocab)

stoi = {token:idx for idx, token in enumerate(vocab)}
itos = {idx:token for token, idx in stoi.items()}
print(stoi)
def encode_word(word, merges):
    pieces = list(word)
    for pair in merges:
        i = 0
        while i < len(pieces) - 1:
            if (pieces[i], pieces[i+1]) == pair:
                pieces[i] = pieces[i] + pieces[i+1]
                pieces.pop(i+1)
            else:
                i += 1
    return pieces

def encode(word, merges, stoi):
    pieces = encode_word(word, merges)
    return [stoi[piece] for piece in pieces]

def decode(token_ids, itos):
    tokens = [itos[idx] for idx in token_ids]
    return ''.join(tokens)

token_ids = encode('lowerstlo', merges, stoi)
print(token_ids)
print(decode(token_ids, itos))

merges: [('l', 'o'), ('lo', 'w'), ('low', 'e'), ('lowe', 'r')]
final vocab: ['e', 'l', 'o', 'r', 's', 't', 'w', 'lo', 'low', 'lowe', 'lower']
{'e': 0, 'l': 1, 'o': 2, 'r': 3, 's': 4, 't': 5, 'w': 6, 'lo': 7, 'low': 8, 'lowe': 9, 'lower': 10}
[10, 4, 5, 7]
lowerstlo


## BPE: First Implementation

We started with a small corpus and represented every word as individual
characters.

For each iteration, the algorithm:

1. Counts all adjacent token pairs across the corpus.
2. Selects the most frequent pair.
3. Merges that pair into a new token.
4. Repeats the process for a fixed number of merges.

For our corpus, the learned merges were:

- `l + o` → `lo`
- `lo + w` → `low`
- `low + e` → `lowe`
- `lowe + r` → `lower`

This produces a vocabulary containing both the original characters and
the newly learned subword tokens.

We then used the learned merge rules to tokenize new words. The tokenizer
starts from characters and applies the learned merges in their learned
order.

For example:

`lowerstlo` → `lower + s + t + lo`

The resulting pieces are then mapped to token IDs using `stoi`, and
`itos` can reconstruct the pieces during decoding.

This demonstrated the complete basic BPE idea: frequent character
patterns can be progressively merged into reusable subword tokens.

### What I understand now

BPE training learns **merge rules**, not just a final vocabulary. The
merge order matters because later merges can operate on tokens created by
earlier merges.

During encoding, the learned merge rules are applied to a new sequence
to determine its subword pieces, which are then converted into token IDs.

### test on unknown/new words below

In [230]:
# let's understand the algorithm manually -> Next experiment: richer corpus
corpus = [
    "low",
    "lower",
    "lowest",
    "playing",
    "played",
    "player",
    "walking",
    "talking",
    "flower",
    "slower",
    "umber"
]

vocab = sorted(set(''.join(corpus)))
words = []
for word in corpus:
    w = []
    for char in word:
        w.append(char)
    words.append(w)

def get_pair_counts(words):
    pairs = {} 
    for word in words:
        for i,j in zip(word, word[1:]):
            pairs[(i,j)] = pairs.get((i,j), 0) + 1
    # print('pairs:', pairs)
    return pairs

def get_best_pair(pair_counts: dict):
    return max(pair_counts.items(), key=lambda item: item[1])[0]

def merge_pair(words, pair):
    new_words = []
    for word in words:
        merged = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i+1]) == pair:
                merged.append(word[i]+word[i+1])
                i += 2
            else:
                merged.append(word[i])
                i += 1
        new_words.append(merged)
    return new_words
            
def train(words, num_merges: int):
    merges = []
    vocab = sorted(set(token for word in words for token in word))
    for _ in range(num_merges):
        pair_counts = get_pair_counts(words)
        best_pair = get_best_pair(pair_counts)
        merges.append(best_pair)
        words = merge_pair(words, best_pair)
        vocab.append(''.join(best_pair))
    return merges, vocab


def encode_word(word, merges):
    pieces = list(word)
    for pair in merges:
        i = 0
        while i < len(pieces) - 1:
            if (pieces[i], pieces[i+1]) == pair:
                pieces[i] = pieces[i] + pieces[i+1]
                pieces.pop(i+1)
            else:
                i += 1
    return pieces

def encode(word, merges, stoi):
    pieces = encode_word(word, merges)
    return [stoi[piece] for piece in pieces]

def decode(token_ids, itos):
    tokens = [itos[idx] for idx in token_ids]
    return ''.join(tokens)


# test on unknown or new words
test_words = [
    "lowering",
    "flowers",
    "flowing",
    "walkers",
    "talkers",
    "playfully",
    "slowly",
]
for num_merges in [20]:
    print(f'\n\nWhen num_merges = {num_merges}')
    merges, vocab = train(words, num_merges)

    stoi = {token:idx for idx, token in enumerate(vocab)}
    itos = {idx:token for token, idx in stoi.items()}

    l = []
    for word in test_words:
        token_ids = encode(word, merges, stoi)
        l.append(len(token_ids))
        print(f'{word} -> {encode(word, merges, stoi)} -> {decode(encode(word, merges, stoi), itos)}')

    print(f'vocab length: {len(vocab)} ||| avg token per word: {round(sum(l)/len(l),2)}')
    print(merges)



When num_merges = 20
lowering -> [22, 27] -> lowering
flowers -> [4, 22, 14] -> flowers
flowing -> [4, 20, 27] -> flowing
walkers -> [17, 29, 21, 14] -> walkers
talkers -> [15, 29, 21, 14] -> talkers
playfully -> [25, 4, 16, 8, 8, 18] -> playfully
slowly -> [14, 20, 8, 18] -> slowly
vocab length: 39 ||| avg token per word: 3.71
[('l', 'o'), ('lo', 'w'), ('e', 'r'), ('low', 'er'), ('p', 'l'), ('pl', 'a'), ('pla', 'y'), ('i', 'n'), ('in', 'g'), ('a', 'l'), ('al', 'k'), ('alk', 'ing'), ('low', 'e'), ('lowe', 's'), ('lowes', 't'), ('play', 'ing'), ('play', 'e'), ('playe', 'd'), ('play', 'er'), ('w', 'alking')]
